# fracburgers - Colab Runner (minimal)

Run your existing scripts on Colab GPU with minimal setup.

### Quick start
1. Runtime -> Change runtime type -> T4 GPU
2. Run Cells 1-5 once
3. Edit params and run any section

---
## Setup

In [12]:
# ── Cell 1: detect environment ───────────────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules
print('Colab' if IN_COLAB else 'Local')


Colab


In [ ]:
# ── Cell 2: locate or clone repo + install ───────────────────────────────
import os, subprocess, sys
from pathlib import Path

if IN_COLAB:
# if False:
    REPO_URL = os.environ.get('FRACBURGERS_REPO_URL', 'https://github.com/wasness-apma/apma2070.git').strip()
    PROJECT_DIR = Path('/content/fracburgers')

    if not (PROJECT_DIR / 'scripts').exists():
        # Inject GitHub token if stored in Colab secrets (key icon → "GITHUB_TOKEN")
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
            # splice token into https URL: https://<token>@github.com/...
            auth_url = REPO_URL.replace('https://', f'https://{token}@')
        except Exception:
            auth_url = REPO_URL  # public repo or token not set

        subprocess.run(
            ['git', 'clone', '--depth', '1', auth_url, str(PROJECT_DIR)],
            check=True,
        )

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_DIR), '-q'], check=True)
else:
    _cwd = Path.cwd()
    PROJECT_DIR = next((p for p in [_cwd, *_cwd.parents] if (p / 'scripts').exists()), None)
    if PROJECT_DIR is None:
        raise RuntimeError(f'Could not find scripts/ starting from {_cwd}.')

os.chdir(PROJECT_DIR)
SCRIPTS_DIR = PROJECT_DIR / 'scripts'
OUT_DIR = PROJECT_DIR / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_DIR : {PROJECT_DIR}')
print(f'scripts     : {[p.name for p in sorted(SCRIPTS_DIR.glob("*.py"))]}')
print(f'results     : {OUT_DIR}')

RuntimeError: Could not find scripts/ starting from /content/fracburgers.

In [10]:
# ── Cell 3: GPU check + unified device variable ──────────────────────────
import tensorflow as tf

# Colab/runtime-aware device detection for TensorFlow
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f'Could not set memory growth: {e}')

    device = '/GPU:0'
    print(f'GPU available: {gpus[0]}')
else:
    device = '/CPU:0'
    print('GPU not available, using CPU')

print(f'Using device: {device}')

GPU available: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
Using device: /GPU:0


In [ ]:
# ── Cell 4: helpers ───────────────────────────────────────────────────────
import subprocess, sys
from pathlib import Path
from IPython.display import Image, Video, display as _disp

def run_script(name: str, params: dict) -> bool:
    args = [sys.executable, '-u', str(SCRIPTS_DIR / name)]
    for k, v in params.items():
        flag = '--' + k.replace('_', '-')
        if isinstance(v, bool):
            if v:
                args.append(flag)
        elif v is not None:
            args += [flag, str(v)]
    print('Running:', ' '.join(str(a) for a in args[2:]))
    ok = subprocess.run(args).returncode == 0
    print('done' if ok else 'FAILED')
    return ok

def show(*paths, width: int = 900):
    for p in paths:
        p = Path(str(p))
        if p.exists():
            _disp(Image(str(p), width=width))
            if IN_COLAB:
                from google.colab import files as _f
                _f.download(str(p))
        else:
            print(f'[not found] {p}')

def show_video(*paths):
    for p in paths:
        p = Path(str(p))
        if p.exists():
            _disp(Video(str(p), embed=True))
            if IN_COLAB:
                from google.colab import files as _f
                _f.download(str(p))
        else:
            print(f'[not found] {p}')

---
## 1. Spectral solve — `solve.py`

Runs the Cole–Hopf spectral solver for one IC and a list of α values.  
Produces one diagnostic figure per α (θ + u spacetimes, mass conservation) and an optional movie.

In [ ]:
# ── Edit parameters ───────────────────────────────────────────────────────
solve_params = dict(
    ic         = 'sine',           # 'sine' | 'gaussian'
    alpha_list = '0.25,0.5,0.75',  # produces one figure per alpha
    nu         = 0.1,
    t_max      = 2.0,
    n_times    = 8,
    N          = 512,
    L          = 3.14159265358979,
    out        = OUT_DIR / 'solve/solution.png',
    # movie    = OUT_DIR / 'solve/movie.gif',  # uncomment to generate animation
    # movie_fps = 15,
)

In [ ]:
run_script('solve.py', solve_params)
show(*sorted((OUT_DIR / 'solve').glob('*.png')))

---
## 2. Train PINN — `train_pinn.py`

Trains a heat-equation PINN `θ(x,t)` with the Cole–Hopf back-transform giving `u`.  
The entire training step (sampling + forward + backward) is compiled into a single `@tf.function` — the GPU runs continuously with no Python overhead in the hot loop.

**GPU tip:** larger `n_collocation` fills the GPU more fully; 16 384 is a good default for a T4.

In [ ]:
# ── Edit parameters ───────────────────────────────────────────────────────
pinn_params = dict(
    ic            = 'sine',
    alpha         = 0.5,
    nu            = 0.1,
    t_max         = 2.0,
    N             = 512,
    L             = 3.14159265358979,
    hidden_layers = '128,128,128,128',  # wider for GPU
    activation    = 'tanh',
    epochs        = 20_000,
    lr            = 1e-3,
    lr_decay      = True,     # cosine decay → lr/100
    n_collocation = 16_384,   # T4: ~2× faster than 8192 per epoch
    n_initial     = 1024,
    pde_weight    = 1.0,
    ic_weight     = 100.0,
    log_every     = 500,
    seed          = 0,
    out_dir       = OUT_DIR / 'pinn',
    device        = device,
)

In [ ]:
run_script('train_pinn.py', pinn_params)
show(
    OUT_DIR / 'pinn/loss.png',
    OUT_DIR / 'pinn/comparison.png',
)

---
## 3. Compare PINN vs spectral — `compare_methods.py`

Loads saved PINN weights and runs a high-resolution spectral reference solve,
then produces a four-panel comparison figure and per-time error metrics.

> **Note:** set `pinn_checkpoint` to the `.weights.h5` file written by step 2.

In [ ]:
# ── Edit parameters ───────────────────────────────────────────────────────
compare_params = dict(
    ic              = 'sine',
    alpha           = 0.5,
    nu              = 0.1,
    t_max           = 2.0,
    n_times         = 21,
    N               = 512,
    N_ref           = 2048,
    L               = 3.14159265358979,
    pinn_checkpoint = OUT_DIR / 'pinn/model.weights.h5',
    out_dir         = OUT_DIR / 'compare',
    # movie         = OUT_DIR / 'compare/movie.gif',
)

In [ ]:
run_script('compare_methods.py', compare_params)
show(*sorted((OUT_DIR / 'compare').glob('*.png')))

---
## 4. Spectral convergence study — `reference_convergence.py`

Sweeps N (grid resolution), α, k, and t to show relative L² / L∞ convergence  
against the closed-form cosine-mode reference.  
`b ≈ a` makes the series ratio `|r(0)| ≈ 0.87` — a genuinely tough solution.

In [ ]:
# ── Edit parameters ───────────────────────────────────────────────────────
conv_params = dict(
    a          = 1.0,
    b          = 0.99,
    nu         = 0.1,
    alpha_list = '0.25,0.5,0.75',
    k_list     = '1,2',
    N_list     = '32,64,128,256,512,1024',
    times      = '0.01,0.1,0.5,1.0',
    n_terms    = 300,
    out_dir    = OUT_DIR / 'convergence',
)

In [ ]:
run_script('reference_convergence.py', conv_params)
show(*sorted((OUT_DIR / 'convergence').glob('*.png')))

---
## 5. Reference solution plots — `plot_reference.py`

Evaluates the closed-form reference `u(x,t) = a + b cos(ωx)` solution  
and produces a grid with rows = times, columns = α values.

In [ ]:
# ── Edit parameters ───────────────────────────────────────────────────────
ref_params = dict(
    a          = 1.0,
    b          = 0.99,
    nu         = 0.1,
    k          = 1.0,
    L          = 3.14159265358979,
    N          = 512,
    n_terms    = 300,
    alpha_list = '0.1,0.3,0.5,0.7,0.9',
    times      = '0.01,0.1,0.5,1.0,2.0',
    out_dir    = OUT_DIR / 'reference',
    # movie      = OUT_DIR / 'reference/movie.gif',
    # movie_frames = 150,
    # movie_fps    = 20,
)

In [ ]:
run_script('plot_reference.py', ref_params)
show(*sorted((OUT_DIR / 'reference').glob('*.png')))
# show_video(OUT_DIR / 'reference/movie.gif')  # uncomment if movie was generated

---
## 6. Diffusion vs dispersion — `plot_diffusion_dispersion.py`

For the linearised equation `u_t = −ν(ik)^α u`, the exact solution is

$$u(x,t) = e^{-\gamma t}\cos(kx - ct), \quad
\gamma = \nu|k|^\alpha\cos\!\left(\frac{\alpha\pi}{2}\right), \quad
c = \nu|k|^\alpha\sin\!\left(\frac{\alpha\pi}{2}\right).$$

Grid: **rows = α**, **columns = k**.  Light → dark = early → late time.

In [ ]:
# ── Edit parameters ───────────────────────────────────────────────────────
disp_params = dict(
    nu         = 0.1,
    alpha_list = '0.1,0.3,0.5,0.7,0.9',
    k_list     = '1,2,4',
    L          = 3.14159265358979,
    N          = 512,
    times      = '0.0,0.25,0.5,1.0,2.0',
    out_dir    = OUT_DIR / 'diffdisp',
    # movie      = OUT_DIR / 'diffdisp/movie.gif',
    # movie_frames = 150,
    # movie_fps    = 20,
)

In [ ]:
run_script('plot_diffusion_dispersion.py', disp_params)
show(OUT_DIR / 'diffdisp/diffusion_dispersion.png')
# show_video(OUT_DIR / 'diffdisp/movie.gif')

In [ ]:
download_results()